# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes (as attributes, not dict subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}, Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all available **record sets** (`@id`) and list the **fields** (column `@id`s) for each, as described in the dataset schema.

In [ ]:
# The Croissant dataset exposes `record_sets` and their schemas for exploration

from pprint import pprint

record_sets_info = {rs['@id']: rs for rs in dataset.metadata.get('record_sets', [])}

if not record_sets_info:
    # Fallback: Try to infer record sets from mlcroissant
    try:
        record_sets_info = {rs['@id']: rs for rs in dataset.metadata.to_json().get('recordSet', [])}
    except Exception:
        record_sets_info = {}

if not record_sets_info:
    # Try mlcroissant Dataset.records_schema
    record_sets_info = {rs['@id']: rs for rs in dataset.records_schema}

print("Record sets and their fields:")
all_fields = {}
for idx, (record_set_id, record_set) in enumerate(record_sets_info.items()):
    print(f"{idx+1}. Record set @id: {record_set_id}")
    # Find fields/columns
    cols = record_set.get('field', record_set.get('column', []))
    if isinstance(cols, dict):
        cols = [cols]
    field_ids = []
    for f in cols:
        if isinstance(f, str):
            field_ids.append(f)
        elif isinstance(f, dict):
            field_ids.append(f.get('@id', str(f)))
        else:
            field_ids.append(str(f))
    if not field_ids:
        # Try for mlcroissant: fields is a list
        if 'fields' in record_set:
            field_ids = [f.get('@id', str(f)) for f in record_set['fields']]
    print(f"   Fields/columns @id: {field_ids}")
    all_fields[record_set_id] = field_ids

if not record_sets_info:
    print("  No explicit record sets found. Trying fallback: list available records.")
    # Try to find a default record set (typical mlcroissant fallback)
    sample = list(dataset.records())
    print(f"  Found default records with fields: {list(sample[0].keys()) if len(sample) else 'no records found.'}")
    record_sets_info = {'default': {}}
    all_fields['default'] = list(sample[0].keys()) if sample else []


## 3. Data Extraction
Load data from selected record set(s) into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Based on the above, build a list of available record_set @ids

record_set_ids = list(all_fields.keys())
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")
        continue
    if records:
        df = pd.DataFrame(records)
        print(f"  Columns for {record_set_id}: {df.columns.tolist()}")
        dataframes[record_set_id] = df
    else:
        print(f"  No records for {record_set_id}")

# For demonstration, pick the first available record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set for analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print("No available record sets to process.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following examples assume the presence of a numeric field such as `cr:peptide_count` and a grouping field such as `cr:accession` (use the correct `@id` for your schema).
Modify the variable assignments below to map to the actual available fields as printed above.

In [ ]:
# Set these as appropriate for your data (using their @ids or exact DataFrame column names)
numeric_field = None
group_field = None

# Example: Let's try to pick a likely numeric and group field from the loaded columns
df = dataframes[main_record_set_id]
print("Sample columns:", df.columns.tolist())
sample_numeric_fields = [col for col in df.columns if any(key in col.lower() for key in ['count', 'abundance', 'coverage', 'mw', 'number', 'peptide'])]
if sample_numeric_fields:
    numeric_field = sample_numeric_fields[0]
    print(f"Using '{numeric_field}' as numeric field.")
else:
    # Fallback: pick the first float/int column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            print(f"Using '{numeric_field}' as numeric field.")
            break
if not numeric_field:
    print("No obvious numeric field found for filtering. Please set 'numeric_field'.")

# For grouping, try accession or description
possible_group = [col for col in df.columns if 'accession' in col.lower() or 'description' in col.lower()]
if possible_group:
    group_field = possible_group[0]
else:
    # Fallback: just skip grouping
    group_field = None

if numeric_field:
    # Replace non-numeric values with NaN
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean()  # Filter: above-average
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
    print(filtered_df.head())

    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        # Show top 10 groups by mean value
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(10,4))
        group_means.plot(kind='bar')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to use [`mlcroissant`](https://pypi.org/project/mlcroissant/) to load, explore, and analyze a dataset described by a Croissant schema from FAIR^2. We performed schema discovery, loaded records by `@id`, filtered and normalized numeric fields, grouped data, and created visual summaries.

**Next steps:**
- Inspect and engineer features for downstream ML tasks
- Run statistical or machine learning analysis
- Explore additional record sets if available
